# Description

In this notebook, I will run the python code (generated by LLM) and evaluate the accuracy of generated code passing test case

In [ ]:
import os
import pandas as pd
import numpy as np
import load_dotenv
import io
import re
import ast
import contextlib
import types
import unittest
import importlib
import sys

from utils.evaluation_utils import run_tests

In [ ]:
# Load df generated code 
path_csv_file = "data/generated_code_gpt-4o.csv"
df = pd.read_csv(path_csv_file)

print(f"Shape of the dataframe: {df.shape}")
df.sample()

### Test single sample

In [ ]:
idx = np.random.randint(0, len(df))

generated_code = df.loc[idx, "generated_code"]
test_case = df.loc[idx, "test_case"]
list_libs = df.loc[idx, "libs"]
# convert string representation of list back to list
list_libs = ast.literal_eval(list_libs)

In [ ]:
summary = run_tests(
    generated_code,
    test_case,
    libs=list_libs,
    expected_signature=("task_func", ["mean", "std_dev", "n"])  # binds/aliases to task_func
)

print(summary["output"])
print(f"Success: {summary['wasSuccessful']}, "
      f"Ran: {summary['testsRun']}, "
      f"Failures: {summary['failures']}, Errors: {summary['errors']}")

percentage_passed = (summary['testsRun'] - summary['failures'] - summary['errors']) / summary['testsRun'] * 100
print(f"Percentage of tests passed: {percentage_passed:.2f}%")

### Run through whole data

In [6]:
list_percentages = []

for idx in range(len(df)):
    if idx % 10 == 0:
        print(f"Evaluating index {idx}/{len(df)}")
    
    try:
        generated_code = df.loc[idx, "generated_code"]
        test_case = df.loc[idx, "test_case"]
        list_libs = df.loc[idx, "libs"]
        list_libs = ast.literal_eval(list_libs) 

        summary = run_tests(
            generated_code,
            test_case,
            libs=list_libs,
            expected_signature=("task_func", ["mean", "std_dev", "n"])  # binds/aliases to task_func
        )
        
        percentage_passed = (summary['testsRun'] - summary['failures'] - summary['errors']) / summary['testsRun'] * 100
        list_percentages.append(percentage_passed)
    except Exception as e:
        list_percentages.append(0.0)  # Assume 0% if error occurs
    
average_percentage = np.mean(list_percentages)
print(f"Average percentage of tests passed over the dataset: {average_percentage:.2f}%")